- Add the standard scaler step: see any difference (Is it better than before?)
- Try using a different model (Is it better than before?) : Decision Tree?
- Save the best model

# Import Libraries

In [ ]:
# Installing required packages
!pip install pyspark
!pip install findspark

In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession,SQLContext

# Create Spark Context

In [ ]:
spark = SparkSession \
    .builder \
    .appName("ML_Classifications") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
sqlContext = SQLContext(sc)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving drybeans.csv to drybeans.csv


In [ ]:
file='drybeans.csv'
df = spark.read.csv(file,header='true',inferSchema=True)

In [ ]:
df.columns

['Area',
 'Perimeter',
 'MajorAxisLength',
 'MinorAxisLength',
 'AspectRation',
 'Eccentricity',
 'ConvexArea',
 'EquivDiameter',
 'Extent',
 'Solidity',
 'roundness',
 'Compactness',
 'ShapeFactor1',
 'ShapeFactor2',
 'ShapeFactor3',
 'ShapeFactor4',
 'Class']

In [ ]:
df.printSchema()

root
 |-- Area: integer (nullable = true)
 |-- Perimeter: double (nullable = true)
 |-- MajorAxisLength: double (nullable = true)
 |-- MinorAxisLength: double (nullable = true)
 |-- AspectRation: double (nullable = true)
 |-- Eccentricity: double (nullable = true)
 |-- ConvexArea: integer (nullable = true)
 |-- EquivDiameter: double (nullable = true)
 |-- Extent: double (nullable = true)
 |-- Solidity: double (nullable = true)
 |-- roundness: double (nullable = true)
 |-- Compactness: double (nullable = true)
 |-- ShapeFactor1: double (nullable = true)
 |-- ShapeFactor2: double (nullable = true)
 |-- ShapeFactor3: double (nullable = true)
 |-- ShapeFactor4: double (nullable = true)
 |-- Class: string (nullable = true)



In [ ]:
df.describe().toPandas().transpose()

,0,1,2,3,4
summary,count,mean,stddev,min,max
Area,13611,53048.284549261625,29324.09571688207,20420,254616
Perimeter,13611,855.2834585996654,214.28969589196151,524.736,1985.37
MajorAxisLength,13611,320.1418673032194,85.6941859593335,183.601165,738.8601535
MinorAxisLength,13611,202.2707140828817,44.97009129411471,122.5126535,460.1984968
AspectRation,13611,1.5832419790188144,0.24667845568580432,1.024867596,2.430306447
Eccentricity,13611,0.750894929372346,0.09200176320620888,0.218951263,0.911422968
ConvexArea,13611,53768.20020571596,29774.915817000012,20684,263261
EquivDiameter,13611,253.06421992490445,59.17712014871156,161.2437642,569.3743583
Extent,13611,0.7497327873564055,0.049086366843964224,0.555314717,0.866194641


In [ ]:
df.select(["Area","Perimeter","Solidity","roundness","Compactness","Class"]).show(5)

+-----+---------+-----------+-----------+-----------+-----+
| Area|Perimeter|   Solidity|  roundness|Compactness|Class|
+-----+---------+-----------+-----------+-----------+-----+
|28395|  610.291|0.988855999|0.958027126|0.913357755|SEKER|
|28734|  638.018|0.984985603|0.887033637|0.953860842|SEKER|
|29380|   624.11|0.989558774|0.947849473|0.908774239|SEKER|
|30008|  645.884|0.976695743|0.903936374|0.928328835|SEKER|
|30140|  620.134| 0.99089325|0.984877069|0.970515523|SEKER|
+-----+---------+-----------+-----------+-----------+-----+
only showing top 5 rows



In [ ]:
df.groupBy('Class').count().orderBy('count').show()

+--------+-----+
|   Class|count|
+--------+-----+
|  BOMBAY|  522|
|BARBUNYA| 1322|
|    CALI| 1630|
|   HOROZ| 1928|
|   SEKER| 2027|
|    SIRA| 2636|
|DERMASON| 3546|
+--------+-----+



In [ ]:
# Convert Class column from string to numerical values
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="Class", outputCol="label") #now, class is a label because we want to predict the class based on features
df = indexer.fit(df).transform(df)

#  Classification

In [ ]:
from pyspark.sql import DataFrameNaFunctions
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler, StringIndexer, VectorIndexer, StandardScaler
from pyspark.ml.classification import LogisticRegression,DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [ ]:
featureColumns =df.columns[:-2]

add Scaler, see any difference?

In [ ]:
from pyspark.ml.classification import LogisticRegression
assembler = VectorAssembler(inputCols=featureColumns, outputCol="features")
scaler= StandardScaler(inputCol= "features", outputCol= "scaled_features")
lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline fit, add scaler to the stages as well

In [ ]:
from pyspark.ml import Pipeline
# Update the pipeline stages to remove the indexer
pipeline = Pipeline(stages=[assembler,scaler,lr])

In [ ]:
(trainingData, testData) = df.randomSplit([0.8,0.2], seed = 13234)
model = pipeline.fit(trainingData)

## Data Tuning
Data tuning means setting different combinations of model parameters (values learnt during training) to automatically search for the best configuration that gives the highest model performance (e.g., accuracy, AUC, F1)

It helps the model learn more effectively and avoid underfitting or overfitting.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

paramGrid = ParamGridBuilder() \
    .addGrid(lr.fitIntercept, [False, True]) \
    .addGrid(lr.maxIter, [5, 10,20]) \
    .build()

three validation sets (numFolds=3)

- It tries different combinations of hyperparameters (from paramGrid)
- Trains a model for each combination using k-fold cross-validation (in this case, 3 folds)
- Evaluate each model using an evaluation metric (default: accuracy)
- Pick the best model and return it as cvModel

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
crossval = CrossValidator(estimator=pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=MulticlassClassificationEvaluator(),
                          numFolds=3)  # use 3+ folds in practice

# Run cross-validation, and choose the best set of parameters.
cvModel = crossval.fit(df)

In [ ]:
cvModel.avgMetrics

[np.float64(0.10766331838117842),
 np.float64(0.10934475875666787),
 np.float64(0.8493139842088375),
 np.float64(0.870634672473035),
 np.float64(0.9208138051090721),
 np.float64(0.9240500079922415)]

which one to use from the 6 parameters? use the best one which is 0.924...

In [ ]:
(trainin, testData) = df.randomSplit([0.8,0.2], seed = 13234 )

## Predictions

In [ ]:
predictions = cvModel.transform(testData)

In [ ]:
predictions.show()

+-----+---------+---------------+---------------+------------+------------+----------+-------------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+--------+-----+--------------------+--------------------+--------------------+--------------------+----------+
| Area|Perimeter|MajorAxisLength|MinorAxisLength|AspectRation|Eccentricity|ConvexArea|EquivDiameter|     Extent|   Solidity|  roundness|Compactness|ShapeFactor1|ShapeFactor2|ShapeFactor3|ShapeFactor4|   Class|label|            features|     scaled_features|       rawPrediction|         probability|prediction|
+-----+---------+---------------+---------------+------------+------------+----------+-------------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+--------+-----+--------------------+--------------------+--------------------+--------------------+----------+
|21397|  535.436|    192.5302973|    141.6521869| 1.359176314| 0.67

In [ ]:
predictions.select("probability","prediction", "label").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-----+
|probability                                                                                                                                                |prediction|label|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-----+
|[0.9999998359091599,3.478591330724456E-8,1.2930476578655286E-7,1.6036512969095182E-13,8.028961673941997E-19,6.24454822704042E-16,1.904937097666127E-17]    |0.0       |0.0  |
|[0.9999998328507358,1.5335591231998537E-7,1.3781006423765188E-8,1.2344923806025495E-11,3.3176295127231414E-18,5.349674287498619E-16,1.1026296204884181E-17]|0.0       |0.0  |
|[0.9999998079483983,1.8996346002840215E-7,2.0400391284252694E-9,4.810267493688298E-11,2.900275789759831E-18,1.30208280287430

We see the added column called "scaled features" again, just like in regression

In [ ]:
prediction_save=predictions.select("rawprediction","probability","prediction", "label").show()

+--------------------+--------------------+----------+-----+
|       rawprediction|         probability|prediction|label|
+--------------------+--------------------+----------+-----+
|[25.3816687120331...|[0.99999983590915...|       0.0|  0.0|
|[24.7665871059535...|[0.99999983285073...|       0.0|  0.0|
|[25.5024541435003...|[0.99999980794839...|       0.0|  0.0|
|[24.8609224426015...|[0.99999980818560...|       0.0|  0.0|
|[25.0281826576623...|[0.99999986719267...|       0.0|  0.0|
|[25.3978068613354...|[0.99999816293330...|       0.0|  0.0|
|[23.4844736093773...|[0.99999295248042...|       0.0|  0.0|
|[24.5699294171865...|[0.99999982026748...|       0.0|  0.0|
|[23.5550655516892...|[0.99999951728286...|       0.0|  0.0|
|[23.0742745990767...|[0.99999904283225...|       0.0|  0.0|
|[23.3747775749716...|[0.99999907920625...|       0.0|  0.0|
|[23.4571787719050...|[0.99999885343047...|       0.0|  0.0|
|[23.1324093359987...|[0.99999821191066...|       0.0|  0.0|
|[22.9200450347279...|[0

save prediction

In [ ]:
predictions.select("prediction", "label").write.save(path="predictions",
                                                     format="com.databricks.spark.csv",
                                                     header='true')

# Try using a different model (Decision Tree)

In [ ]:
dt = DecisionTreeClassifier(labelCol="label",featuresCol="scaled_features") #label = Class

In [ ]:
pipeline_dt = Pipeline(stages=[assembler, scaler, dt])

In [ ]:
(trainingData_dt, testData_dt) = df.randomSplit([0.8, 0.2], seed=13234)

model fit

In [ ]:
model_tree = pipeline_dt.fit(trainingData_dt)

predict

In [ ]:
prediction_tree = model_tree.transform(testData_dt)

In [ ]:
prediction_tree.show()

+-----+---------+---------------+---------------+------------+------------+----------+-------------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+--------+-----+--------------------+--------------------+--------------------+--------------------+----------+
| Area|Perimeter|MajorAxisLength|MinorAxisLength|AspectRation|Eccentricity|ConvexArea|EquivDiameter|     Extent|   Solidity|  roundness|Compactness|ShapeFactor1|ShapeFactor2|ShapeFactor3|ShapeFactor4|   Class|label|            features|     scaled_features|       rawPrediction|         probability|prediction|
+-----+---------+---------------+---------------+------------+------------+----------+-------------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+--------+-----+--------------------+--------------------+--------------------+--------------------+----------+
|21397|  535.436|    192.5302973|    141.6521869| 1.359176314| 0.67

## Do Evalution to compare decision tree with logistic regression, which one is better?

### Start with Logistic regress

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

In [ ]:
def evaluate(result):
    predictionAndLabels = result.select("prediction", "label")
    metrics = ["f1","precisionByLabel","recallByLabel","weightedPrecision","weightedRecall","accuracy"]
    for m in metrics:
        evaluator = MulticlassClassificationEvaluator(metricName=m)
        print(str(m) + ": " + str(evaluator.evaluate(predictionAndLabels)))

In [ ]:
evaluate(predictions)

f1: 0.9313256552535362
precisionByLabel: 0.9242424242424242
recallByLabel: 0.9217032967032966
weightedPrecision: 0.9315289213535639
weightedRecall: 0.9313974591651544
accuracy: 0.9313974591651543


In [ ]:
# Evaluate model performance
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Accuracy =", accuracy)

Accuracy = 0.9313974591651543


In [ ]:
prediction=predictions.select("prediction", "label")

In [ ]:
metrics = MulticlassMetrics(prediction.rdd.map(tuple))

/usr/local/lib/python3.11/dist-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
metrics.confusionMatrix().toArray().transpose()

array([[671.,  51.,   3.,   1.,   0.,   0.,   0.],
       [ 48., 468.,  14.,   1.,   3.,   7.,   0.],
       [  7.,   8., 399.,   0.,   0.,   2.,   0.],
       [  2.,  11.,   0., 365.,   3.,   0.,   0.],
       [  0.,   2.,   1.,   3., 319.,  14.,   0.],
       [  0.,   0.,   4.,   1.,   3., 239.,   0.],
       [  0.,   0.,   0.,   0.,   0.,   0., 105.]])

### Decision Tree

In [ ]:
evaluate(prediction_tree)

f1: 0.8853239480986017
precisionByLabel: 0.9004092769440655
recallByLabel: 0.9065934065934066
weightedPrecision: 0.8864270432942818
weightedRecall: 0.8856624319419236
accuracy: 0.8856624319419237


### Conclusion
Logic Regression model is CLEARLY better becuase the scores for f1, precision, accuracy, etc are greater than Decision Tree's.
Especially F1, F1 Score balances precision and recall : it tells you how well the model avoids both false positives and false negatives

Meaning :
- It’s more accurate
- It has better precision & recall
- It’s more balanced

### Save better model (Already saved)

In [ ]:
sc.stop()